In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

LLM_MODEL = "distilbert-base-uncased"
MAX_TOKEN_LENGTH = 512
data_submission_path = "submission.csv"

In [20]:


# Local path configurations
train_data_path = "kaggle/input/competitions/llm-classification-finetuning/train.csv"
test_data_path = "kaggle/input/competitions/llm-classification-finetuning/test.csv"
model_save_path = "./model"


# #Kaggle path configurations
# train_data_path = "/kaggle/input/competitions/llm-classification-finetuning/train.csv"
# test_data_path = "/kaggle/input/competitions/llm-classification-finetuning/test.csv"
# model_save_path = "/kaggle/input/models/yanxinye/llm-classifier-nicole-ye/other/default/1"


# To submit to Kaggle:
# 1. Manually upload the trained model to Kaggle's input -> Add model -> Upload model, and then copy the Kaggle path configurations above to load the model in the inference notebook.
# 2. Add the competition dataset to Kaggle's input, then use the Kaggle path configurations to load the data.
# 3. Turn off internet access in Kaggle's notebook settings to avoid any issues with loading the model or data.


In [21]:
def maybe_swap(row):
    if np.random.rand() < 0.5:
        row["response_a"], row["response_b"] = row["response_b"], row["response_a"]
        if row["labels"] == 0:
            row["labels"] = 1
        elif row["labels"] == 1:
            row["labels"] = 0
    return row

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_TOKEN_LENGTH,
    )

def truncate_text(text, max_tokens):
    ids = tokenizer(
        str(text),
        truncation=True,
        max_length=max_tokens,
        add_special_tokens=False
    )["input_ids"]
    return tokenizer.decode(ids)

def build_text(row):
    prompt = truncate_text(row["prompt"], 128)
    a = truncate_text(row["response_a"], 192)
    b = truncate_text(row["response_b"], 192)

    return (
        "[PROMPT]\n" + prompt +
        "\n\n[RESPONSE A]\n" + a +
        "\n\n[RESPONSE B]\n" + b
    )

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained(model_save_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_save_path)

model.eval()

Using device: cpu


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 23540.62it/s]


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [23]:
# 0) Load test data
test_df = pd.read_csv(test_data_path)


# 3) Build text field
if set(["prompt", "response_a", "response_b"]).issubset(test_df.columns):
    test_df["text"] = test_df.apply(build_text, axis=1)
else:
    raise ValueError("Missing one of prompt/response_a/response_b")

# 4) HF dataset
test_dataset = Dataset.from_pandas(test_df[["text"]])

# 6) tokenize
test_dataset = test_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.remove_columns([c for c in test_dataset.column_names if c not in ["input_ids", "attention_mask", "text"]])



Map: 100%|██████████| 3/3 [00:00<00:00, 1177.95 examples/s]


In [24]:
from transformers import DataCollatorWithPadding

# IMPORTANT: remove raw text before DataLoader
test_dataset = test_dataset.remove_columns(["text"])


data_collator = DataCollatorWithPadding(tokenizer)

from torch.utils.data import DataLoader

dataloader = DataLoader(
    test_dataset,
    batch_size=16,
    collate_fn=data_collator   # ✅ THIS FIXES EVERYTHING
)

In [25]:
model.eval()

all_logits = []

with torch.no_grad():
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits

        all_logits.append(logits.cpu())

logits = torch.cat(all_logits, dim=0)
probs = torch.softmax(logits, dim=-1).numpy()

In [26]:
print("nan logits?", torch.isnan(logits).any().item())
print("inf logits?", torch.isinf(logits).any().item())



out = pd.DataFrame({
    "id": test_df["id"].values,
    "winner_model_a": probs[:, 0],
    "winner_model_b": probs[:, 1],
    "winner_model_tie": probs[:, 2],
})

out.to_csv(data_submission_path, index=False)
print(out.head())

nan logits? False
inf logits? False
        id  winner_model_a  winner_model_b  winner_model_tie
0   136060        0.209476        0.234906          0.555618
1   211333        0.365254        0.320558          0.314188
2  1233961        0.265310        0.453572          0.281118
